In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path
import os

# Caminho absoluto robusto para o banco de dados, mesmo no Voilà
project_root = Path.cwd().parents[1]
db_path = project_root / "data" / "sjur_recortes.db"

if not db_path.exists():
    raise FileNotFoundError(f"❌ Banco de dados não encontrado: {db_path}")


In [ ]:

def carregar_publicacoes_com_data():
    conn = sqlite3.connect(db_path)
    query = """
    SELECT p.message_id, e.data_processamento
    FROM publicacoes p
    LEFT JOIN emails_processados e ON p.message_id = e.message_id
    ORDER BY e.data_processamento DESC
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

def gerar_links_outlook(df):
    df['link_outlook'] = df['message_id'].apply(lambda mid: f'<a href="outlook:{mid}">{mid[:50]}...</a>')
    df = df.rename(columns={"data_processamento": "Data de Processamento"})
    return df[["Data de Processamento", "link_outlook"]].rename(columns={"link_outlook": "Outlook Link"})

df_raw = carregar_publicacoes_com_data()
df_links = gerar_links_outlook(df_raw)

display(HTML("<h3>📧 Emails já processados</h3>"))
display(HTML(df_links.to_html(escape=False, index=False)))
